In [6]:
# Save all our generated data to one big file to make it easier

import torch
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import StandardScaler
import h5py

# Constants
N_LAYERS = 48
LAYERS = [f"decoder.model.decoder.layers.{n}" for n in range(N_LAYERS)]
METADATA_PATH = "/home/harinit9/orcd/pool/musicgen-data-nokey/dataset_metadata.json"
ACT_BATCHED_PATH = Path("/home/wyf/orcd/pool/musicgen-activations-nokey/activations_pooled")

In [7]:
with open(METADATA_PATH) as fin:
    metadata = json.load(fin)

N_CLIPS = len(metadata)

def get_clip_np(clip: dict):
    """
    Retrieve the activations for a single clip idx
    """
    act_path = clip["activations_path"]
    acts = torch.load(act_path)

    layer_acts = []
    for layer_idx, layer in enumerate(LAYERS):
        # acts[layer] is a list
        # shape: (256, 2048)
        layer_acts.append(torch.cat(acts[layer], dim=1).mean(axis=0).numpy())

    # shape: (N_LAYERS, 256, 2048)
    return np.stack(layer_acts)

X0 = get_clip_np(metadata[0])
print(f"{X0.shape=}")
print(f"{X0.nbytes=:,}")

X0.shape=(48, 256, 2048)
X0.nbytes=100,663,296


In [ ]:
# Save using npy lol
for idx, clip in tqdm(enumerate(tqdm(metadata))):
    clip_acts = get_clip_np(clip)
    np.save(ACT_BATCHED_PATH / f"clip_{clip['clip_id']}.npy", clip_acts)

## Do it layer-wise

In [ ]:
ACTS_BY_LAYER_PATH = Path("/home/wyf/orcd/pool/musicgen-activations-nokey/acts_by_layer")

from numpy.lib.format import open_memmap

def export_layer(layer_idx):
    out_path = ACTS_BY_LAYER_PATH / f"layer_{layer_idx:02d}.npy"

    mm = open_memmap(
        out_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(metadata), 256, 2048),
    )

    for i, clip in enumerate(tqdm(metadata)):
        arr = np.load(ACT_BATCHED_PATH / f"clip_{clip['clip_id']}.npy")[layer_idx]
        mm[i] = arr
        del arr

    mm.flush()

n = 24
print(f"Exporting layer {n}")
export_layer(n)

Exporting layer 8


100%|██████████| 1000/1000 [01:08<00:00, 14.61it/s]
